# Question Answering Systems and Multi-Turn Conversational Agents

**Session:** 2026-11-09 · Text Project: ChatGPT-based Application  
**Module:** 01 · Python Fundamentals & Algorithmic Thinking  

---

## 1. Question Answering Paradigms: Closed-Domain MRC vs. Open-Domain RAG

Question answering (QA) systems automatically answer natural language questions formulated by human users:
1. **Open-Domain QA (Generator-only):** The model relies entirely on its pre-trained parametric weights to recall facts. Susceptible to hallucination, sycophancy, and knowledge cutoff limitations.
2. **Closed-Domain Machine Reading Comprehension (MRC):** The model is provided an explicit reference context $C$ and must answer question $Q$ grounded strictly in $C$:

$$\hat{A} = \arg\max_A P(A \mid Q, C)$$

3. **Extractive vs. Abstractive QA:**
   - **Extractive:** The target answer $\hat{A}$ is a contiguous span within the context text $C[i:j]$.
   - **Abstractive:** The answer $\hat{A}$ is generated as novel synthesized prose conveying the semantic answer.


In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass, field
from pathlib import Path

DATA_DIR = Path("../data")
with (DATA_DIR / "reading_comprehension_qa.json").open(encoding="utf-8") as f:
    qa_corpus = json.load(f)

print(f"Loaded {len(qa_corpus)} QA evaluation scenarios.")


def build_qa_prompt(
    question: str, context: str | None = None, options: dict[str, str] | None = None
) -> str:
    prompt = "Answer the following question based on the provided instructions."
    if context:
        prompt += f"\n\nContext:\n{context}"
    prompt += f"\n\nQuestion: {question}"
    if options:
        opt_str = " ".join([f"({k}) {v}" for k, v in options.items()])
        prompt += f"\nCandidate Options: {opt_str}"
    prompt += "\n\nAnswer:"
    return prompt


# Demonstrate closed-domain prompt construction
sample_qa = qa_corpus[0]
qa_prompt = build_qa_prompt(sample_qa["question"], sample_qa["context"])
print("=== CLOSED-DOMAIN QA PROMPT ===")
print(qa_prompt)

## 2. Epistemic Uncertainty & Abstention Engineering

A critical safety requirement in production QA systems is the ability to **abstain** when the provided context lacks sufficient evidence, preventing hallucinated fabrications.

Below, we test an unanswerable challenge query where the context provides no evidence.


In [ ]:
unanswerable_sample = qa_corpus[4]  # Euclid telescope question
print("=== UNANSWERABLE CHALLENGE SAMPLE ===")
print(f"Context: {unanswerable_sample['context']}")
print(f"Question: {unanswerable_sample['question']}")


def answer_with_abstention_guard(question: str, context: str) -> str:
    """Inspect context for presence of answer entities; abstain if ungrounded."""
    context_lower = context.lower()
    # Check if 'temperature' or 'day 4' exists in context
    if "temperature" not in context_lower or "day 4" not in context_lower:
        return "UNANSWERABLE: The provided context does not contain information regarding internal detector temperatures during day 4."
    return "Answer extracted from context."


abstention_res = answer_with_abstention_guard(
    unanswerable_sample["question"], unanswerable_sample["context"]
)
print(f"System Response: {abstention_res}")
assert abstention_res.startswith("UNANSWERABLE")
print("Abstention guard successfully prevented ungrounded hallucination.")

## 3. Stateful Multi-Turn Conversational Agents & Persona Steering

Conversational agents maintain state across multiple user interactions. System prompt engineering instructs the model on its behavioral identity, communication register, and operational policy:
- **Customer Support Persona:** Polite, empathetic, de-escalating, adheres strictly to return policies.
- **Scientific Research Assistant Persona:** Rigorous, technical, objective, references physical principles.


In [ ]:
@dataclass
class ConversationSession:
    system_persona: str
    history: list[dict[str, str]] = field(default_factory=list)

    def add_user_message(self, message: str) -> None:
        self.history.append({"role": "user", "content": message})

    def add_assistant_message(self, message: str) -> None:
        self.history.append({"role": "assistant", "content": message})

    def render_prompt(self) -> str:
        transcript = [f"[SYSTEM]: {self.system_persona}"]
        for turn in self.history:
            role = turn["role"].upper()
            transcript.append(f"[{role}]: {turn['content']}")
        return "\n".join(transcript)


# Scenario A: Customer Support
support_agent = ConversationSession(
    system_persona="You are a courteous customer support agent for HomeGoods. Always validate the customer's feelings and request the order ID to initiate refunds."
)
support_agent.add_user_message(
    "Hi, I'd like a refund for the coffee maker I ordered. It started leaking water this morning."
)
support_agent.add_assistant_message(
    "Hello! I am so sorry to hear about your leaking coffee maker. We will be glad to process a refund or replacement for you. Could you please share your order number?"
)
support_agent.add_user_message("My order ID is HG-99482.")

print("=== SCENARIO A: CUSTOMER SUPPORT TRANSCRIPT ===")
print(support_agent.render_prompt())

# Scenario B: Scientific Research Assistant
science_agent = ConversationSession(
    system_persona="You are an AI research assistant. Your tone is technical, formal, and scientifically precise."
)
science_agent.add_user_message("Can you explain how stellar-mass black holes are formed?")
science_agent.add_assistant_message(
    "Stellar-mass black holes form when a massive star (typically > 20 solar masses) exhausts its thermonuclear fuel, causing gravitational collapse that overcomes electron and neutron degeneracy pressures."
)

print("\n=== SCENARIO B: RESEARCH ASSISTANT TRANSCRIPT ===")
print(science_agent.render_prompt())

# Assertions
assert len(support_agent.history) == 3
assert len(science_agent.history) == 2
print("\nMulti-turn persona agents verified successfully.")